### Imports and device

In [10]:
import time
import torch
import matplotlib.pyplot as plt
import itertools
import torch.nn as nn
import torch.optim as optim

from data_preprocessing import create_dataloader, denormalize
from models import Generator, PatchGANDiscriminator

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4090


### Dataset paths

In [2]:
from pathlib import Path

DATASET_ROOT = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/dataset/"
)

MONET_SOURCE = DATASET_ROOT / "monet_jpg"
PHOTO_SOURCE = DATASET_ROOT / "photo_jpg"
SPLIT_ROOT = DATASET_ROOT / "split_data"

print("Dataset root:", DATASET_ROOT)
print("Monet source:", MONET_SOURCE)
print("Photo source:", PHOTO_SOURCE)
print("Split root:", SPLIT_ROOT)

print("Monet folder exists:", MONET_SOURCE.exists())
print("Photo folder exists:", PHOTO_SOURCE.exists())
print("Split folder exists:", SPLIT_ROOT.exists())

Dataset root: /app/DATA266-Lab1-Pair45/task3_GAN/dataset
Monet source: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/monet_jpg
Photo source: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/photo_jpg
Split root: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data
Monet folder exists: True
Photo folder exists: True
Split folder exists: True


### Split the data

In [3]:
# import random
# import shutil


# def split_images(
#     source_dir,
#     domain_name,
#     output_root,
#     train_ratio=0.8,
#     validation_ratio=0.1,
#     seed=42
# ):
#     image_extensions = {
#         ".jpg",
#         ".jpeg",
#         ".png",
#         ".bmp",
#         ".webp"
#     }

#     image_paths = sorted([
#         path for path in source_dir.rglob("*")
#         if path.is_file()
#         and path.suffix.lower() in image_extensions
#     ])

#     if len(image_paths) == 0:
#         raise ValueError(
#             f"No images found in: {source_dir}"
#         )

#     rng = random.Random(seed)
#     rng.shuffle(image_paths)

#     total_images = len(image_paths)

#     train_end = int(total_images * train_ratio)

#     validation_end = int(
#         total_images
#         * (train_ratio + validation_ratio)
#     )

#     split_paths = {
#         "train": image_paths[:train_end],

#         "validation": image_paths[
#             train_end:validation_end
#         ],

#         "test": image_paths[
#             validation_end:
#         ]
#     }

#     for split_name, paths in split_paths.items():

#         destination_dir = (
#             output_root
#             / split_name
#             / domain_name
#         )

#         destination_dir.mkdir(
#             parents=True,
#             exist_ok=True
#         )

#         for image_path in paths:
#             shutil.copy2(
#                 image_path,
#                 destination_dir / image_path.name
#             )

#         print(
#             f"{domain_name} | "
#             f"{split_name}: {len(paths)} images"
#         )


# # Split photos independently
# split_images(
#     source_dir=PHOTO_SOURCE,
#     domain_name="photos",
#     output_root=SPLIT_ROOT,
#     seed=42
# )

# # Split Monet images independently
# split_images(
#     source_dir=MONET_SOURCE,
#     domain_name="monet",
#     output_root=SPLIT_ROOT,
#     seed=43
# )

photos | train: 5630 images
photos | validation: 704 images
photos | test: 704 images
monet | train: 240 images
monet | validation: 30 images
monet | test: 30 images


### Create the DataLoader

In [3]:
BATCH_SIZE = 16

train_loader = create_dataloader(
    domain_a_dir=SPLIT_ROOT / "train" / "photos",
    domain_b_dir=SPLIT_ROOT / "train" / "monet",
    batch_size=BATCH_SIZE,
    train=True,
    shuffle=True,
    num_workers=0
)

print("Batch size:", BATCH_SIZE)
print("Number of training batches:", len(train_loader))

Batch size: 16
Number of training batches: 352


### Create the four CycleGAN networks

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

G_AB = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

G_BA = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

D_A = PatchGANDiscriminator(
    input_channels=3
).to(device)

D_B = PatchGANDiscriminator(
    input_channels=3
).to(device)

print("Models created successfully.")
print("Device:", device)

Models created successfully.
Device: cuda


### Load the best previous checkpoint

In [5]:
BEST_CHECKPOINT = (
    Path("/app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi")
    / "checkpoints"
    / "best_cyclegan.pth"
)

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {BEST_CHECKPOINT.resolve()}")

checkpoint = torch.load(BEST_CHECKPOINT, map_location=device)
G_AB.load_state_dict(checkpoint["G_AB"])
G_BA.load_state_dict(checkpoint["G_BA"])
D_A.load_state_dict(checkpoint["D_A"])
D_B.load_state_dict(checkpoint["D_B"])

print("Loaded checkpoint epoch:", checkpoint.get("epoch", "unknown"))

Loaded checkpoint epoch: 53


### Define losses and Optimizers

In [8]:
criterion_gan = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

lambda_cycle = 10.0
lambda_identity = 5.0

# Smaller learning rate for fine-tuning
FINE_TUNE_LR = 5e-5
BETA1 = 0.5
BETA2 = 0.999

optimizer_G = optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()),
    lr=FINE_TUNE_LR,
    betas=(BETA1, BETA2)
)
optimizer_D_A = optim.Adam(
    D_A.parameters(), lr=FINE_TUNE_LR, betas=(BETA1, BETA2)
)
optimizer_D_B = optim.Adam(
    D_B.parameters(), lr=FINE_TUNE_LR, betas=(BETA1, BETA2)
)

print("Fine-tuning learning rate:", FINE_TUNE_LR)


Fine-tuning learning rate: 5e-05


### 

### Fine-tune and save the best fine-tuned checkpoint

In [12]:
import os
import time
import itertools
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# Load the best previous checkpoint
# ============================================================

BEST_CHECKPOINT = (
    Path("/app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi")
    / "checkpoints"
    / "best_cyclegan.pth"
)

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {BEST_CHECKPOINT}"
    )

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device
)

G_AB.load_state_dict(checkpoint["G_AB"])
G_BA.load_state_dict(checkpoint["G_BA"])
D_A.load_state_dict(checkpoint["D_A"])
D_B.load_state_dict(checkpoint["D_B"])

print(
    "Loaded best checkpoint from epoch:",
    checkpoint.get("epoch", "unknown"),
    flush=True
)


# ============================================================
# Fine-tuning settings
# ============================================================

NUM_FINE_TUNE_EPOCHS = 15

FINE_TUNE_LR = 0.00005
BETA1 = 0.5
BETA2 = 0.999

FINE_TUNE_DIR = Path("checkpoints_finetuned")
FINE_TUNE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Fine-tuning learning rate:",
    FINE_TUNE_LR,
    flush=True
)

print(
    "Fine-tuning checkpoint folder:",
    FINE_TUNE_DIR.resolve(),
    flush=True
)


# ============================================================
# Loss functions
# ============================================================

criterion_gan = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

lambda_cycle = 10.0
lambda_identity = 5.0


# ============================================================
# Fresh fine-tuning optimizers
# ============================================================

optimizer_G = optim.Adam(
    itertools.chain(
        G_AB.parameters(),
        G_BA.parameters()
    ),
    lr=FINE_TUNE_LR,
    betas=(BETA1, BETA2)
)

optimizer_D_A = optim.Adam(
    D_A.parameters(),
    lr=FINE_TUNE_LR,
    betas=(BETA1, BETA2)
)

optimizer_D_B = optim.Adam(
    D_B.parameters(),
    lr=FINE_TUNE_LR,
    betas=(BETA1, BETA2)
)


# ============================================================
# AMP setup
# ============================================================

use_amp = torch.cuda.is_available()

scaler = torch.cuda.amp.GradScaler(
    enabled=use_amp
)

print(
    "AMP enabled:",
    use_amp,
    flush=True
)


# ============================================================
# Helper functions
# ============================================================

def set_requires_grad(models, requires_grad):

    for model in models:
        for parameter in model.parameters():
            parameter.requires_grad = requires_grad


def safe_torch_save(obj, path):

    temporary_path = str(path) + ".tmp"

    torch.save(
        obj,
        temporary_path
    )

    os.replace(
        temporary_path,
        path
    )


# ============================================================
# History and monitoring
# ============================================================

history_finetune = {
    "generator": [],
    "discriminator_A": [],
    "discriminator_B": [],
    "cycle": [],
    "identity": []
}

best_monitor = float("inf")

training_start = time.time()


# ============================================================
# Training start
# ============================================================

print("\nFine-tuning started.", flush=True)
print(
    f"Total epochs: {NUM_FINE_TUNE_EPOCHS}",
    flush=True
)
print(
    f"Batches per epoch: {len(train_loader)}",
    flush=True
)
print(
    f"Device: {device}",
    flush=True
)
print("-" * 80, flush=True)


# ============================================================
# Fine-tuning loop
# ============================================================

for epoch in range(NUM_FINE_TUNE_EPOCHS):

    epoch_start = time.time()

    total_loss_G = 0.0
    total_loss_D_A = 0.0
    total_loss_D_B = 0.0
    total_loss_cycle = 0.0
    total_loss_identity = 0.0

    number_of_batches = 0

    print(
        f"\nStarting Epoch "
        f"[{epoch + 1}/{NUM_FINE_TUNE_EPOCHS}]",
        flush=True
    )


    # --------------------------------------------------------
    # Process every batch
    # --------------------------------------------------------

    for batch_index, batch in enumerate(
        train_loader,
        start=1
    ):

        real_A = batch["A"].to(
            device,
            non_blocking=True
        )

        real_B = batch["B"].to(
            device,
            non_blocking=True
        )


        # ====================================================
        # Step 1: Train generators
        # ====================================================

        set_requires_grad(
            [D_A, D_B],
            False
        )

        optimizer_G.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            # Photo → Monet
            fake_B = G_AB(real_A)

            # Monet → Photo
            fake_A = G_BA(real_B)

            # Photo → Monet → Photo
            reconstructed_A = G_BA(fake_B)

            # Monet → Photo → Monet
            reconstructed_B = G_AB(fake_A)

            # Identity mappings
            identity_A = G_BA(real_A)
            identity_B = G_AB(real_B)

            # Discriminator predictions
            prediction_fake_B_for_G = D_B(fake_B)
            prediction_fake_A_for_G = D_A(fake_A)

            # Adversarial generator losses
            loss_G_AB = criterion_gan(
                prediction_fake_B_for_G,
                torch.ones_like(
                    prediction_fake_B_for_G
                )
            )

            loss_G_BA = criterion_gan(
                prediction_fake_A_for_G,
                torch.ones_like(
                    prediction_fake_A_for_G
                )
            )

            # Cycle-consistency loss
            loss_cycle_A = criterion_cycle(
                reconstructed_A,
                real_A
            )

            loss_cycle_B = criterion_cycle(
                reconstructed_B,
                real_B
            )

            loss_cycle = (
                loss_cycle_A
                + loss_cycle_B
            )

            # Identity loss
            loss_identity_A = criterion_identity(
                identity_A,
                real_A
            )

            loss_identity_B = criterion_identity(
                identity_B,
                real_B
            )

            loss_identity = (
                loss_identity_A
                + loss_identity_B
            )

            # Total generator loss
            loss_G = (
                loss_G_AB
                + loss_G_BA
                + lambda_cycle * loss_cycle
                + lambda_identity * loss_identity
            )


        scaler.scale(
            loss_G
        ).backward()

        scaler.step(
            optimizer_G
        )


        # ====================================================
        # Step 2: Train Photo discriminator D_A
        # ====================================================

        set_requires_grad(
            [D_A, D_B],
            True
        )

        optimizer_D_A.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            prediction_real_A = D_A(real_A)

            prediction_fake_A = D_A(
                fake_A.detach()
            )

            loss_D_A_real = criterion_gan(
                prediction_real_A,
                torch.ones_like(
                    prediction_real_A
                )
            )

            loss_D_A_fake = criterion_gan(
                prediction_fake_A,
                torch.zeros_like(
                    prediction_fake_A
                )
            )

            loss_D_A = 0.5 * (
                loss_D_A_real
                + loss_D_A_fake
            )


        scaler.scale(
            loss_D_A
        ).backward()

        scaler.step(
            optimizer_D_A
        )


        # ====================================================
        # Step 3: Train Monet discriminator D_B
        # ====================================================

        optimizer_D_B.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            prediction_real_B = D_B(real_B)

            prediction_fake_B = D_B(
                fake_B.detach()
            )

            loss_D_B_real = criterion_gan(
                prediction_real_B,
                torch.ones_like(
                    prediction_real_B
                )
            )

            loss_D_B_fake = criterion_gan(
                prediction_fake_B,
                torch.zeros_like(
                    prediction_fake_B
                )
            )

            loss_D_B = 0.5 * (
                loss_D_B_real
                + loss_D_B_fake
            )


        scaler.scale(
            loss_D_B
        ).backward()

        scaler.step(
            optimizer_D_B
        )


        # Update AMP scaling
        scaler.update()


        # ====================================================
        # Store losses
        # ====================================================

        total_loss_G += loss_G.item()
        total_loss_D_A += loss_D_A.item()
        total_loss_D_B += loss_D_B.item()
        total_loss_cycle += loss_cycle.item()
        total_loss_identity += loss_identity.item()

        number_of_batches += 1


        # ====================================================
        # Print every batch
        # ====================================================

        print(
            f"Epoch [{epoch + 1}/{NUM_FINE_TUNE_EPOCHS}] | "
            f"Batch [{batch_index}/{len(train_loader)}] | "
            f"Generator: {loss_G.item():.4f} | "
            f"D_A: {loss_D_A.item():.4f} | "
            f"D_B: {loss_D_B.item():.4f} | "
            f"Cycle: {loss_cycle.item():.4f} | "
            f"Identity: {loss_identity.item():.4f}",
            flush=True
        )


        # Release unnecessary references
        del (
            real_A,
            real_B,
            fake_A,
            fake_B,
            reconstructed_A,
            reconstructed_B,
            identity_A,
            identity_B,
            prediction_fake_A_for_G,
            prediction_fake_B_for_G,
            prediction_real_A,
            prediction_real_B,
            prediction_fake_A,
            prediction_fake_B
        )


    # ========================================================
    # Calculate epoch averages
    # ========================================================

    if number_of_batches == 0:

        print(
            "No batches were processed. Training stopped.",
            flush=True
        )

        break


    average_loss_G = (
        total_loss_G
        / number_of_batches
    )

    average_loss_D_A = (
        total_loss_D_A
        / number_of_batches
    )

    average_loss_D_B = (
        total_loss_D_B
        / number_of_batches
    )

    average_loss_cycle = (
        total_loss_cycle
        / number_of_batches
    )

    average_loss_identity = (
        total_loss_identity
        / number_of_batches
    )


    # ========================================================
    # Store epoch history
    # ========================================================

    history_finetune["generator"].append(
        average_loss_G
    )

    history_finetune["discriminator_A"].append(
        average_loss_D_A
    )

    history_finetune["discriminator_B"].append(
        average_loss_D_B
    )

    history_finetune["cycle"].append(
        average_loss_cycle
    )

    history_finetune["identity"].append(
        average_loss_identity
    )


    # ========================================================
    # Epoch summary
    # ========================================================

    monitor_score = (
        average_loss_cycle
        + average_loss_identity
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    print("\nEpoch summary:", flush=True)

    print(
        f"Epoch [{epoch + 1}/{NUM_FINE_TUNE_EPOCHS}] | "
        f"Time: {epoch_time:.2f} seconds",
        flush=True
    )

    print(
        f"Average Generator loss: "
        f"{average_loss_G:.6f}",
        flush=True
    )

    print(
        f"Average Photo discriminator loss: "
        f"{average_loss_D_A:.6f}",
        flush=True
    )

    print(
        f"Average Monet discriminator loss: "
        f"{average_loss_D_B:.6f}",
        flush=True
    )

    print(
        f"Average Cycle-consistency loss: "
        f"{average_loss_cycle:.6f}",
        flush=True
    )

    print(
        f"Average Identity loss: "
        f"{average_loss_identity:.6f}",
        flush=True
    )

    print(
        f"Monitor score: "
        f"{monitor_score:.6f}",
        flush=True
    )


    # ========================================================
    # Prepare checkpoint
    # ========================================================

    checkpoint_out = {
        "epoch": epoch + 1,
        "G_AB": G_AB.state_dict(),
        "G_BA": G_BA.state_dict(),
        "D_A": D_A.state_dict(),
        "D_B": D_B.state_dict(),
        "optimizer_G": optimizer_G.state_dict(),
        "optimizer_D_A": optimizer_D_A.state_dict(),
        "optimizer_D_B": optimizer_D_B.state_dict(),
        "history": history_finetune,
        "monitor_score": monitor_score
    }


    # ========================================================
    # Save latest checkpoint
    # ========================================================

    latest_path = (
        FINE_TUNE_DIR
        / "latest_finetuned.pth"
    )

    safe_torch_save(
        checkpoint_out,
        latest_path
    )

    print(
        "Latest checkpoint saved:",
        latest_path,
        flush=True
    )


    # ========================================================
    # Save best checkpoint
    # ========================================================

    if monitor_score < best_monitor:

        best_monitor = monitor_score

        best_path = (
            FINE_TUNE_DIR
            / "best_finetuned.pth"
        )

        safe_torch_save(
            checkpoint_out,
            best_path
        )

        print(
            "Best fine-tuned checkpoint saved.",
            flush=True
        )

    else:

        print(
            "Best checkpoint not improved.",
            flush=True
        )

    print("-" * 80, flush=True)


# ============================================================
# Training completed
# ============================================================

total_training_time = (
    time.time()
    - training_start
)

print(
    "\nFine-tuning completed.",
    flush=True
)

print(
    f"Total fine-tuning time: "
    f"{total_training_time / 60:.2f} minutes",
    flush=True
)

print(
    "Latest checkpoint:",
    FINE_TUNE_DIR / "latest_finetuned.pth",
    flush=True
)

print(
    "Best checkpoint:",
    FINE_TUNE_DIR / "best_finetuned.pth",
    flush=True
)

Loaded best checkpoint from epoch: 53
Fine-tuning learning rate: 5e-05
Fine-tuning checkpoint folder: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/checkpoints_finetuned
AMP enabled: True

Fine-tuning started.
Total epochs: 15
Batches per epoch: 352
Device: cuda
--------------------------------------------------------------------------------

Starting Epoch [1/15]
Epoch [1/15] | Batch [1/352] | Generator: 2.9219 | D_A: 0.2203 | D_B: 0.1158 | Cycle: 0.1388 | Identity: 0.1188
Epoch [1/15] | Batch [2/352] | Generator: 2.8292 | D_A: 0.2331 | D_B: 0.1331 | Cycle: 0.1493 | Identity: 0.1231
Epoch [1/15] | Batch [3/352] | Generator: 2.6378 | D_A: 0.1904 | D_B: 0.1241 | Cycle: 0.1274 | Identity: 0.1082
Epoch [1/15] | Batch [4/352] | Generator: 2.6101 | D_A: 0.1902 | D_B: 0.1093 | Cycle: 0.1150 | Identity: 0.0983
Epoch [1/15] | Batch [5/352] | Generator: 2.2499 | D_A: 0.2177 | D_B: 0.1162 | Cycle: 0.1016 | Identity: 0.0853
Epoch [1/15] | Batch [6/352] | Generator: 2.7010 | D_A: 0.2045 | D_B: 

KeyboardInterrupt: 

### Loading th best fine tuned checkpoint

In [13]:
from pathlib import Path
import torch

best_finetuned_path = Path("checkpoints_finetuned/best_finetuned.pth")

checkpoint = torch.load(
    best_finetuned_path,
    map_location=device
)

G_AB.load_state_dict(checkpoint["G_AB"])
G_BA.load_state_dict(checkpoint["G_BA"])
D_A.load_state_dict(checkpoint["D_A"])
D_B.load_state_dict(checkpoint["D_B"])

G_AB.to(device).eval()
G_BA.to(device).eval()
D_A.to(device).eval()
D_B.to(device).eval()

print("Loaded fine-tuned checkpoint.")
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Monitor score:", checkpoint.get("monitor"))

Loaded fine-tuned checkpoint.
Checkpoint epoch: 1
Monitor score: None


In [18]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class FolderDataset(Dataset):

    def __init__(self, folder):

        self.paths = sorted([
            path for path in Path(folder).rglob("*")
            if path.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            }
        ])

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5)
            )
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):

        image = Image.open(
            self.paths[index]
        ).convert("RGB")

        return self.transform(image)


def save_tensor_image(image, output_path):

    image = image.detach().cpu()
    image = image * 0.5 + 0.5
    image = image.clamp(0, 1)

    image_array = (
        image.permute(1, 2, 0).numpy() * 255
    ).astype("uint8")

    Image.fromarray(image_array).save(
        output_path,
        format="JPEG",
        quality=95
    )

print("FolderDataset is ready.")

FolderDataset is ready.


### Generate translated test images using the best checkpoint For Monet to Photo that is A2B

In [19]:
# Read all Monet images
monet_dataset = FolderDataset(MONET_SOURCE)

monet_loader = DataLoader(
    monet_dataset,
    batch_size=10,
    shuffle=False,
    num_workers=0
)

# Your naming convention:
# pred_A2B = Monet → Photo
MONET_TO_PHOTO = Path(
    "kaggle_finetuning_outputs/pred_A2B"
)

MONET_TO_PHOTO.mkdir(
    parents=True,
    exist_ok=True
)

G_BA.eval()

counter = 0

with torch.no_grad():

    for monets in monet_loader:

        monets = monets.to(device)

        # Monet → Photo
        generated_photos = G_BA(monets)

        for image in generated_photos:

            save_tensor_image(
                image,
                MONET_TO_PHOTO
                / f"monet_to_photo_{counter:05d}.jpg"
            )

            counter += 1

            if counter == 300:
                break

        if counter == 300:
            break

print(
    "Monet → Photo images generated:",
    counter
)

print(
    "Saved in:",
    MONET_TO_PHOTO.resolve()
)

Monet → Photo images generated: 300
Saved in: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_finetuning_outputs/pred_A2B


### Generate translated test images using the best checkpoint For Photo to Monet that is B2A


In [20]:
# Read Photo images
photo_dataset = FolderDataset(PHOTO_SOURCE)

photo_loader = DataLoader(
    photo_dataset,
    batch_size=10,
    shuffle=False,
    num_workers=0
)

# Your naming convention:
# pred_B2A = Photo → Monet
PHOTO_TO_MONET = Path(
    "kaggle_finetuning_outputs/pred_B2A"
)

PHOTO_TO_MONET.mkdir(
    parents=True,
    exist_ok=True
)

G_AB.eval()

counter = 0

with torch.no_grad():

    for photos in photo_loader:

        photos = photos.to(device)

        # Photo → Monet
        generated_monets = G_AB(photos)

        for image in generated_monets:

            save_tensor_image(
                image,
                PHOTO_TO_MONET
                / f"photo_to_monet_{counter:05d}.jpg"
            )

            counter += 1

            if counter == 300:
                break

        if counter == 300:
            break

print(
    "Photo → Monet images generated:",
    counter
)

print(
    "Saved in:",
    PHOTO_TO_MONET.resolve()
)

Photo → Monet images generated: 300
Saved in: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_finetuning_outputs/pred_B2A


### Calculate the distribution metrics: FID and KID in both directions.

In [22]:
from pathlib import Path

# According to your current naming:
# pred_A2B = Monet → Photo
# pred_B2A = Photo → Monet

PHOTO_TO_MONET_V2 = Path(
    "kaggle_finetuning_outputs/pred_B2A"
)

MONET_TO_PHOTO_V2 = Path(
    "kaggle_finetuning_outputs/pred_A2B"
)

print(
    "Photo → Monet folder exists:",
    PHOTO_TO_MONET_V2.exists()
)

print(
    "Photo → Monet images:",
    len(list(PHOTO_TO_MONET_V2.glob("*.jpg")))
)

print(
    "Monet → Photo folder exists:",
    MONET_TO_PHOTO_V2.exists()
)

print(
    "Monet → Photo images:",
    len(list(MONET_TO_PHOTO_V2.glob("*.jpg")))
)

Photo → Monet folder exists: True
Photo → Monet images: 300
Monet → Photo folder exists: True
Monet → Photo images: 300


In [23]:
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance


# --------------------------------------------------
# Device and paths
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SPLIT_ROOT = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data"
)

# According to your naming convention:
# pred_B2A = Photo → Monet
# pred_A2B = Monet → Photo

PHOTO_TO_MONET_V2 = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/"
    "kaggle_finetuning_outputs/pred_B2A"
)

MONET_TO_PHOTO_V2 = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/"
    "kaggle_finetuning_outputs/pred_A2B"
)

print("Photo → Monet folder exists:", PHOTO_TO_MONET_V2.exists())
print(
    "Photo → Monet images:",
    len(list(PHOTO_TO_MONET_V2.glob("*.jpg")))
)

print("Monet → Photo folder exists:", MONET_TO_PHOTO_V2.exists())
print(
    "Monet → Photo images:",
    len(list(MONET_TO_PHOTO_V2.glob("*.jpg")))
)


# --------------------------------------------------
# Image loading
# --------------------------------------------------

image_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.PILToTensor()
])


def get_image_paths(folder):

    return sorted([
        path
        for path in Path(folder).rglob("*")
        if path.is_file()
        and path.suffix.lower() in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        }
    ])


def load_images(image_paths, batch_size=8):

    batches = []

    for start in range(0, len(image_paths), batch_size):

        batch_paths = image_paths[
            start:start + batch_size
        ]

        images = []

        for path in batch_paths:

            image = Image.open(path).convert("RGB")
            image = image_transform(image)
            images.append(image)

        batches.append(torch.stack(images))

    return batches


# --------------------------------------------------
# Metric calculation
# --------------------------------------------------

def calculate_fid_kid(real_dir, generated_dir):

    real_paths = get_image_paths(real_dir)
    generated_paths = get_image_paths(generated_dir)

    print()
    print("Real folder:", real_dir)
    print("Generated folder:", generated_dir)
    print("Real images:", len(real_paths))
    print("Generated images:", len(generated_paths))

    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=False
    ).to(device)

    subset_size = min(
        50,
        len(real_paths),
        len(generated_paths)
    )

    kid = KernelInceptionDistance(
        feature=2048,
        subset_size=subset_size,
        normalize=False
    ).to(device)

    real_batches = load_images(real_paths)
    generated_batches = load_images(generated_paths)

    with torch.no_grad():

        for batch in real_batches:

            batch = batch.to(device)

            fid.update(
                batch,
                real=True
            )

            kid.update(
                batch,
                real=True
            )

        for batch in generated_batches:

            batch = batch.to(device)

            fid.update(
                batch,
                real=False
            )

            kid.update(
                batch,
                real=False
            )

    fid_score = fid.compute().item()

    kid_mean, kid_std = kid.compute()

    return {
        "FID": fid_score,
        "KID_mean": kid_mean.item(),
        "KID_std": kid_std.item()
    }


# --------------------------------------------------
# Photo → Monet
# --------------------------------------------------

photo_to_monet_metrics = calculate_fid_kid(
    real_dir=SPLIT_ROOT / "test" / "monet",
    generated_dir=PHOTO_TO_MONET_V2
)

print()
print("Photo → Monet metrics")
print(photo_to_monet_metrics)


# --------------------------------------------------
# Monet → Photo
# --------------------------------------------------

monet_to_photo_metrics = calculate_fid_kid(
    real_dir=SPLIT_ROOT / "test" / "photos",
    generated_dir=MONET_TO_PHOTO_V2
)

print()
print("Monet → Photo metrics")
print(monet_to_photo_metrics)

Photo → Monet folder exists: True
Photo → Monet images: 300
Monet → Photo folder exists: True
Monet → Photo images: 300

Real folder: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data/test/monet
Generated folder: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_finetuning_outputs/pred_B2A
Real images: 30
Generated images: 300


/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)



Photo → Monet metrics
{'FID': 209.83950805664062, 'KID_mean': 0.01476356666535139, 'KID_std': 0.0034551701974123716}

Real folder: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data/test/photos
Generated folder: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_finetuning_outputs/pred_A2B
Real images: 704
Generated images: 300

Monet → Photo metrics
{'FID': 91.77436065673828, 'KID_mean': 0.020696409046649933, 'KID_std': 0.003957967273890972}
